# Counterfactual Advantage Estimation

Given a branch state $s$ and two sampled continuations $a_A$ (original) and $a_B$ (counterfactual), we:

1. Show the reflector both hindsight trajectories from $s$ and extract forward-looking guidance $g$.
2. Score both actions under two policies:
   - $\pi_\text{base}(a \mid s)$ — the raw actor at the branch state.
   - $\pi_\text{ic}(a \mid s, g)$ — the same actor with $g$ injected into context.
3. Form signed advantage
   $$\Delta = \big[\log \pi_\text{ic}(a_A \mid s, g) - \log \pi_\text{base}(a_A \mid s)\big] - \big[\log \pi_\text{ic}(a_B \mid s, g) - \log \pi_\text{base}(a_B \mid s)\big].$$
4. $\text{sign}(\Delta)$ predicts which action the guided policy prefers. Compare to `preference` (`original` → A, `counterfactual` → B). Report sign-match rate on non-tie pairs.

Scoring uses the **full assistant message** (reasoning + action tag) at `branch_message_index`.

> **vLLM config note.** For the scoring prompts, the base and in-context variants share a long common prefix. If vLLM is launched with `--enable-prefix-caching` (default on in recent versions), the shared prefix is reused across calls and the in-context scoring is essentially free prefill. Confirm with `ps -ef | grep vllm` on the host or by looking for `prefix_cache_hit` in the server logs — this is the single biggest performance lever after prompt length.

In [1]:
import os, json, re, math, random
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer
from tqdm.auto import tqdm

load_dotenv('../configs/vllm.env')

ACTOR_MODEL = os.environ['MODEL_STR']
ACTOR_PORT  = int(os.environ['VLLM_PORT'])
JUDGE_MODEL = os.environ['JUDGE_STR']
JUDGE_PORT  = int(os.environ['JUDGE_PORT'])
VLLM_KEY    = os.environ.get('VLLM_API_KEY', 'local-token')

actor_client = OpenAI(base_url=f'http://localhost:{ACTOR_PORT}/v1', api_key=VLLM_KEY)
judge_client = OpenAI(base_url=f'http://localhost:{JUDGE_PORT}/v1', api_key=VLLM_KEY)

print('actor :', ACTOR_MODEL, 'port', ACTOR_PORT)
print('judge :', JUDGE_MODEL, 'port', JUDGE_PORT)

/home/weiyong/miniconda3/envs/newtonbench/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


actor : Qwen/Qwen3.5-9B port 8002
judge : Qwen/Qwen3.5-35B-A3B-FP8 port 8006


In [2]:
tokenizer = AutoTokenizer.from_pretrained(ACTOR_MODEL)
print('vocab size:', tokenizer.vocab_size)

vocab size: 248044


## Load counterfactual pairs

In [3]:
# Toggle between original judge-based labels and RMSLE-primary labels
USE_RMSLE_LABELS = True  # Set False for original symbolic-equivalence-based labels

label_file = 'counterfactual_pairs_100.rmsle.jsonl' if USE_RMSLE_LABELS else 'counterfactual_pairs_100.jsonl'
df = pd.read_json(label_file, lines=True)
df_eval = df[df['preference'] != 'tie'].reset_index(drop=True)
print(f"Label source: {'RMSLE-primary' if USE_RMSLE_LABELS else 'judge-primary'}")
print(f"Total pairs: {len(df)}, Non-ties: {len(df_eval)}, Ties: {len(df) - len(df_eval)}")
print(f'total={len(df)}  non-tie={len(df_eval)}')
df_eval['preference'].value_counts()



Label source: RMSLE-primary
Total pairs: 100, Non-ties: 45, Ties: 55
total=100  non-tie=45


preference
counterfactual    24
original          21
Name: count, dtype: int64

In [4]:
print(df['equation_difficulty'].value_counts())
print(df['model_system'].value_counts())

print('\nBefore above, after below:\n ')
print(df_eval['equation_difficulty'].value_counts())
df_eval['model_system'].value_counts()

equation_difficulty
easy      70
medium    30
Name: count, dtype: int64
model_system
complex_system      38
simple_system       38
vanilla_equation    24
Name: count, dtype: int64

Before above, after below:
 
equation_difficulty
easy      29
medium    16
Name: count, dtype: int64


model_system
complex_system      22
simple_system       19
vanilla_equation     4
Name: count, dtype: int64

## Hindsight extraction and reflector

The reflector now sees only the realized non-counterfactual future from the original trajectory and emits forward-looking guidance. Anti-leakage controls remain, but the regeneration filter is intentionally lighter to avoid degenerate rewrite loops. `<think>...</think>` blocks are stripped from assistant messages to cut reflector prompt length.

In [5]:
# Toggle between scoring only the action block vs the full assistant text (without CoT)
SCORE_ACTION_BLOCK_ONLY = True  # Set False to score full assistant text with CoT stripped
USE_JOINT_ACTION_REFLECTOR = True  # Set True to show both actions to reflector and ask for comparison

TASK_HEADER = (
    'TASK CONTEXT:\n'
    'The agent is discovering an unknown physical law in a simulated '
    'universe where physics may differ from ours. Budget: 10 experiment '
    'rounds. Final submission is via <final_law>. Measurements are '
    'noise-free and deterministic. The agent may call <run_experiment>, '
    '<python>, or <final_law> (one action per turn).'
)

_THINK_RE = re.compile(r'<think>.*?</think>\s*', flags=re.DOTALL)
_THINK_TAIL_RE = re.compile(r'^.*?</think>\s*', flags=re.DOTALL)
_FINAL_LAW_RE = re.compile(r'<final_law>.*?</final_law>', flags=re.DOTALL)
MASKED_FINAL_LAW = '<final_law>\n[final law intentionally omitted to avoid answer leakage]\n</final_law>'

GUIDANCE_ISSUE_PATTERNS = {
    'equation_like': re.compile(r'(?:def\s+discovered_law|return\s+|\*\*|\^|[A-Za-z_][A-Za-z0-9_]*\s*[=≈~]\s*[-+]?\d)'),
    'final_law_reference': re.compile(r'(?:<final_law>|discovered_law|functional form)', re.I),
}

def strip_think(content: str) -> str:
    content = _THINK_TAIL_RE.sub('', content)
    content = _THINK_RE.sub('', content)
    if content.lstrip().startswith('Thinking Process:'):
        marker = '\n\n'
        idx = content.find(marker)
        if idx != -1:
            content = content[idx + len(marker):]
    return content.strip()

def mask_final_law_blocks(content: str) -> str:
    return _FINAL_LAW_RE.sub(MASKED_FINAL_LAW, content)

def detect_guidance_issues(content: str) -> list:
    issues = [name for name, pattern in GUIDANCE_ISSUE_PATTERNS.items() if pattern.search(content or '')]
    if len((content or '').split()) > 220:
        issues.append('too_long')
    return issues

def _truncate_by_role(role, content, assistant_len=2000, other_len=600):
    limit = assistant_len if role == 'assistant' else other_len
    if len(content) <= limit:
        return content
    return content[:limit] + f'\n... [truncated, {len(content) - limit} chars omitted]'

def _format_messages(messages, skip_system=True, strip_cot=True, mask_final_law=True):
    lines = []
    for msg in messages:
        role = msg.get('role', 'unknown')
        if skip_system and role == 'system':
            continue
        content = msg.get('content', '')
        if strip_cot and role == 'assistant':
            content = strip_think(content)
        if mask_final_law and role == 'assistant':
            content = mask_final_law_blocks(content)
        content = _truncate_by_role(role, content)
        lines.append(f'[{role.upper()}]:\n{content}\n')
    return '\n'.join(lines)

def extract_action_block(content: str, agent_backend: str = '', mask_final_law: bool = True) -> str:
    for tag in ('final_law', 'run_experiment', 'python'):
        m = re.search(fr'<{tag}>.*?</{tag}>', content, flags=re.DOTALL)
        if m:
            block = m.group(0)
            return mask_final_law_blocks(block) if tag == 'final_law' and mask_final_law else block
    return content.strip()[:300]

def extract_assistant_no_think(content: str, mask_final_law: bool = True) -> str:
    """Return the full assistant text with think blocks removed, optionally masking final law."""
    content = strip_think(content)
    if mask_final_law:
        content = mask_final_law_blocks(content)
    return content.strip()

def extract_hindsight_pointwise(pair: dict) -> dict:
    bi = pair['branch_message_index']
    backend = pair.get('agent_backend', 'vanilla_agent')
    ch_o = pair['original']['chat_history']
    ch_c = pair['counterfactual']['chat_history']
    o_content = ch_o[bi].get('content', '')
    c_content = ch_c[bi].get('content', '')
    return {
        'branch_turn': pair['branch_turn'],
        'branch_message_index': bi,
        'state_prefix': ch_o[:bi],
        'suffix_A': ch_o[bi:],
        'suffix_B': ch_c[bi:],
        'branch_msg_A': ch_o[bi],
        'branch_msg_B': ch_c[bi],
        'action_A': extract_action_block(o_content, backend, mask_final_law=True),
        'action_B': extract_action_block(c_content, backend, mask_final_law=True),
        'score_action_A': (extract_action_block(o_content, backend, mask_final_law=False)
                           if SCORE_ACTION_BLOCK_ONLY else extract_assistant_no_think(o_content, mask_final_law=False)),
        'score_action_B': (extract_action_block(c_content, backend, mask_final_law=False)
                           if SCORE_ACTION_BLOCK_ONLY else extract_assistant_no_think(c_content, mask_final_law=False)),
        'agent_backend': backend,
    }

def format_pointwise_prompt(h: dict, state_window: int = 4, randomize_order: bool = True,
                        seed: Optional[int] = None, strip_cot: bool = True,
                        mask_final_law: bool = True):
    lines = [TASK_HEADER, '', '=' * 60, f"STATE AT DECISION POINT (turn {h['branch_turn']})", '=' * 60]
    lines.append(_format_messages(h['state_prefix'][-state_window:], skip_system=True, strip_cot=strip_cot, mask_final_law=mask_final_law))
    lines.append('=' * 60)
    lines.append('REALIZED FUTURE FROM THIS STATE')
    lines.append('=' * 60)
    lines.append(_format_messages(h['suffix_A'], skip_system=True, strip_cot=strip_cot, mask_final_law=mask_final_law))
    return '\n'.join(lines), {'Realized Future': 'A'}

SYSTEM_MSG_DIRECTIVE = (
    'You are a strategic advisor to a scientific-discovery agent. You observe '
    'the state at a decision point and the realized future that followed from '
    'the action taken there. Your job is to produce guidance that helps the '
    'agent make better decisions at this same state.\n\n'
    'CRITICAL: observing a trajectory does not mean it was a good trajectory. '
    'Judge honestly:\n'
    '- Did the experiments resolve real uncertainty, or did the agent '
    'over-commit on thin evidence?\n'
    '- Were there obvious diagnostics the agent skipped?\n'
    '- Was the final submission premature or well-supported?\n\n'
    'Your guidance will be injected before the agent acts. Be concrete and '
    'directional:\n'
    '- Name specific parameter values, regimes, and experiments from the '
    'trajectory. Quote numbers and ranges when relevant.\n'
    '- Describe observed trends qualitatively: monotonicity, scaling '
    'regimes, where data is thin, where the conclusion is thinly supported.\n'
    '- State whether the evidence supports committing or exploring more.\n'
    '- If exploring, name the single most informative experiment.\n\n'
    'YOU MAY NOT:\n'
    '- Write the functional form of the law as an equation.\n'
    '- State fitted exponents or coefficients.\n'
    '- Produce a <final_law> payload or verbatim experiment JSON.\n\n'
    'Under 200 words. Generic heuristics ("verify carefully", "think step '
    'by step") are actively harmful — omit them.'
)

REFLECTOR_USER_MSG_DIRECTIVE = (
    'Write guidance for the agent about to act at the above state. '
    'Critically assess the realized future: did it build a well-supported '
    'conclusion or over-commit on thin evidence? Recommend a specific next '
    'action and target. Be direct about trajectory quality.'
)

def call_pointwise_reflector(prompt_text: str, temperature: float = 0.6, max_attempts: int = 2) -> dict:
    messages = [
        {'role': 'system', 'content': SYSTEM_MSG_DIRECTIVE},
        {'role': 'user', 'content': f'{prompt_text}\n\n{REFLECTOR_USER_MSG_DIRECTIVE}'},
    ]
    last_raw, last_guidance, last_issues = '', '', []
    for attempt in range(1, max_attempts + 1):
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=messages,
            temperature=temperature,
            max_tokens=10000,
        )
        last_raw = resp.choices[0].message.content or ''
        last_guidance = strip_think(last_raw)
        last_issues = detect_guidance_issues(last_guidance)
        if not last_issues:
            return {'guidance': last_guidance, 'raw_guidance': last_raw,
                    'attempts': attempt, 'regenerated': attempt > 1, 'issues': []}
        messages.extend([
            {'role': 'assistant', 'content': last_raw},
            {'role': 'user', 'content': (
                'Rewrite the guidance more simply. Your previous answer violated these constraints: '
                f"{', '.join(last_issues)}. Remove leaked answer content and keep only short strategic guidance."
            )},
        ])
    return {'guidance': last_guidance, 'raw_guidance': last_raw,
            'attempts': max_attempts, 'regenerated': max_attempts > 1, 'issues': last_issues}

SYSTEM_MSG_JOINT_ACTION = (
    'You are a strategic advisor to a scientific-discovery agent at a '
    'decision point. You observe the current state, two candidate actions '
    '(A and B), and the realized future that followed from action A. '
    'Action B was not taken, so you do not observe its future.\n\n'
    'Your job is to compare the two actions and give guidance that helps '
    'the agent choose the better one at this state.\n\n'
    'STRUCTURE YOUR COMPARISON:\n'
    '1. Assess A using the realized future. Did its experiments resolve '
    'real uncertainty, or did A rest on thin or under-probed evidence? '
    'Judge honestly.\n'
    '2. Assess B from its action text alone. What is B probing that the '
    'state has not yet resolved? What assumption does B rest on?\n'
    '3. Take a directional stance. State which action you believe is more '
    'likely to advance correct discovery and why. Do not hedge symmetrically.\n\n'
    'BE CONCRETE:\n'
    '- Name specific parameter values, regimes, and experiments from the '
    'trajectory. Quote numbers and ranges when relevant.\n'
    '- Describe observed trends qualitatively: monotonicity, scaling '
    'regimes, where data is thin, where A\'s conclusion is thinly supported.\n'
    '- Describe each action by what it proposes in concrete terms.\n\n'
    'YOU MAY NOT:\n'
    '- Write the functional form of the law as an equation.\n'
    '- State fitted exponents or coefficients.\n'
    '- Produce a <final_law> payload or verbatim experiment JSON.\n\n'
    'Under 220 words. Generic heuristics ("verify carefully", "think step '
    'by step") are actively harmful — omit them.'
)

REFLECTOR_USER_MSG_JOINT_ACTION = (
    'Compare actions A and B for the agent at the state above. '
    'Assess A from its realized future, assess B from its action text alone, '
    'and state which is more likely to advance correct discovery. '
    'Refer to each action by what it proposes, not by A or B.'
)

def format_joint_action_prompt(h: dict, state_window: int = 4,
                               strip_cot: bool = True,
                               mask_final_law: bool = True) -> str:
    lines = [TASK_HEADER, '', '=' * 60,
             f"STATE AT DECISION POINT (turn {h['branch_turn']})", '=' * 60]
    lines.append(_format_messages(
        h['state_prefix'][-state_window:], skip_system=True,
        strip_cot=strip_cot, mask_final_law=mask_final_law))
    lines += ['', '=' * 60,
              'ACTION A (realized future observed below)', '=' * 60,
              h['action_A']]
    lines += ['', '=' * 60,
              'REALIZED FUTURE FROM ACTION A', '=' * 60,
              _format_messages(h['suffix_A'][1:], skip_system=True,
                               strip_cot=strip_cot,
                               mask_final_law=mask_final_law)]
    lines += ['', '=' * 60,
              'ACTION B (future NOT observed; reason from action text only)',
              '=' * 60, h['action_B']]
    return '\n'.join(lines)


def call_joint_action_reflector(prompt_text: str, temperature: float = 0.6, max_attempts: int = 2) -> dict:
    messages = [
        {'role': 'system', 'content': SYSTEM_MSG_JOINT_ACTION},
        {'role': 'user', 'content': f'{prompt_text}\n\n{REFLECTOR_USER_MSG_JOINT_ACTION}'},
    ]
    last_raw, last_guidance, last_issues = '', '', []
    for attempt in range(1, max_attempts + 1):
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=messages,
            temperature=temperature,
            max_tokens=20000,
        )
        last_raw = resp.choices[0].message.content or ''
        last_guidance = strip_think(last_raw)
        last_issues = detect_guidance_issues(last_guidance)
        if not last_issues:
            return {'guidance': last_guidance, 'raw_guidance': last_raw,
                    'attempts': attempt, 'regenerated': attempt > 1, 'issues': []}
        messages.extend([
            {'role': 'assistant', 'content': last_raw},
            {'role': 'user', 'content': (
                'Rewrite the guidance more simply. Your previous answer violated these constraints: '
                f"{', '.join(last_issues)}. Remove leaked answer content and keep only short strategic guidance."
            )},
        ])
    return {'guidance': last_guidance, 'raw_guidance': last_raw,
            'attempts': max_attempts, 'regenerated': max_attempts > 1, 'issues': last_issues}



## Scoring: log-probs of a branch assistant message under vLLM

Render the chat with the model's tokenizer, then hit vLLM's `/v1/completions` with `echo=True, logprobs=1` to get per-token logprobs of the prompt. Summing over the assistant-message token span gives $\log \pi(a \mid s)$.

- **Base prompt:** messages up to and including the branch assistant turn.
- **In-context prompt:** same, but with guidance $g$ inserted as a user turn immediately before the branch assistant turn.
- **State windowing:** `SCORING_STATE_WINDOW` optionally trims the pre-branch history to the system message(s) plus the last *N* non-system turns. Base and in-context prompts use the same window so $\Delta$ is still well-defined; this is a speed/fidelity knob. `None` keeps the full prefix.

In [6]:
SCORING_STATE_WINDOW: Optional[int] = None  # None = full history; int = system msgs + last N non-system turns

GUIDANCE_WRAPPER = (
    '[STRATEGIC ADVISOR NOTE — forward-looking guidance for your next action]\n'
    '{g}\n'
    '[END NOTE. Use this context as you decide your next action.]'
)

def _window_state_prefix(prefix, window: Optional[int]):
    if window is None:
        return prefix
    system_msgs = [m for m in prefix if m.get('role') == 'system']
    other_msgs  = [m for m in prefix if m.get('role') != 'system']
    if len(other_msgs) <= window:
        return prefix
    return system_msgs + other_msgs[-window:]

def build_prompts(state_prefix, branch_msg, guidance: Optional[str],
                  state_window: Optional[int] = None):
    """Return (full_text, prefix_text); full ends with the assistant branch message."""
    state_prefix = _window_state_prefix(state_prefix, state_window)
    msgs = list(state_prefix)
    if guidance is not None:
        msgs = msgs + [{'role': 'user', 'content': GUIDANCE_WRAPPER.format(g=guidance)}]
    prefix_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full_msgs = msgs + [branch_msg]
    full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False)
    assert full_text.startswith(prefix_text), 'prefix alignment failed — chat template inconsistency'
    return full_text, prefix_text

def score_action_logprob(state_prefix, branch_msg, guidance: Optional[str],
                         state_window: Optional[int] = None) -> dict:
    full_text, prefix_text = build_prompts(state_prefix, branch_msg, guidance, state_window=state_window)
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)['input_ids']
    full_ids   = tokenizer(full_text,   add_special_tokens=False)['input_ids']
    action_len = len(full_ids) - len(prefix_ids)
    if action_len <= 0:
        return {'logprob': float('nan'), 'n_tokens': 0, 'note': 'empty action span'}
    resp = actor_client.completions.create(
        model=ACTOR_MODEL,
        prompt=full_text,
        max_tokens=1,
        temperature=0.0,
        echo=True,
        logprobs=1,
    )
    tok_logprobs = resp.choices[0].logprobs.token_logprobs
    action_lps = [lp for lp in tok_logprobs[-action_len:] if lp is not None]
    return {
        'logprob': float(sum(action_lps)),
        'n_tokens': len(action_lps),
        'mean_logprob': float(sum(action_lps) / max(len(action_lps), 1)),
    }
def score_action_block_logprob(state_prefix, action_block: str, guidance: Optional[str],
                              state_window: Optional[int] = None) -> dict:
    action_msg = {'role': 'assistant', 'content': action_block}
    return score_action_logprob(state_prefix, action_msg, guidance, state_window=state_window)

## Token-length diagnostics

Check prompt lengths before running the full sweep. Shows scoring lengths under both full history and `SCORING_STATE_WINDOW` so you can see the speedup the windowing buys.

In [7]:
MAX_CTX = 115_000

def prompt_token_lengths(pair: dict) -> dict:
    h = extract_hindsight_pointwise(pair)
    prompt_text, _ = format_pointwise_prompt(h, randomize_order=False, seed=0)
    reflector_user = f'{prompt_text}\n\n{REFLECTOR_USER_MSG_DIRECTIVE}'
    reflector_total = len(tokenizer(SYSTEM_MSG_DIRECTIVE + reflector_user, add_special_tokens=False)['input_ids'])

    out = {'pair_index': pair['pair_index'], 'reflector_toks': reflector_total}
    for label, branch_msg in [('A', h['branch_msg_A']), ('B', h['branch_msg_B'])]:
        # full (no windowing)
        full_base, _ = build_prompts(h['state_prefix'], branch_msg, guidance=None, state_window=None)
        out[f'{label}_base_full_toks']    = len(tokenizer(full_base, add_special_tokens=False)['input_ids'])
        # windowed (what the sweep will actually send)
        win_base, _ = build_prompts(h['state_prefix'], branch_msg, guidance=None, state_window=SCORING_STATE_WINDOW)
        win_ic,   _ = build_prompts(h['state_prefix'], branch_msg, guidance='[placeholder]', state_window=SCORING_STATE_WINDOW)
        out[f'{label}_base_win_toks']     = len(tokenizer(win_base, add_special_tokens=False)['input_ids'])
        out[f'{label}_ic_win_toks']       = len(tokenizer(win_ic,   add_special_tokens=False)['input_ids'])
    return out

tlen_rows = [prompt_token_lengths(row.to_dict()) for _, row in df_eval.iterrows()]
tlen_df = pd.DataFrame(tlen_rows)

cols = ['reflector_toks',
        'A_base_full_toks', 'A_base_win_toks', 'A_ic_win_toks',
        'B_base_full_toks', 'B_base_win_toks', 'B_ic_win_toks']
print(f'SCORING_STATE_WINDOW = {SCORING_STATE_WINDOW}  (None = full history)')
print(f'{"":28s} {"min":>7} {"p50":>7} {"p90":>7} {"p99":>7} {"max":>7}  over_{MAX_CTX//1000}k')
for col in cols:
    s = tlen_df[col]
    over = (s > MAX_CTX).sum()
    print(f'{col:28s} {s.min():7.0f} {s.quantile(.5):7.0f} {s.quantile(.9):7.0f} {s.quantile(.99):7.0f} {s.max():7.0f}  {over}')

SCORING_STATE_WINDOW = None  (None = full history)
                                 min     p50     p90     p99     max  over_115k
reflector_toks                  1003    9097   17130   23750   24957  0
A_base_full_toks                6696   19920   45325   56496   61217  0
A_base_win_toks                 6696   19920   45325   56496   61217  0
A_ic_win_toks                   6737   19961   45366   56537   61258  0
B_base_full_toks                6644   19821   47751   58685   63617  0
B_base_win_toks                 6644   19821   47751   58685   63617  0
B_ic_win_toks                   6685   19862   47792   58726   63658  0


## Per-pair advantage — flat executor

One shared `EXEC_POOL` runs every vLLM request. Each pair submits the reflector call first, then scores only the extracted action blocks under the guided policy. This matches the writeup's contrastive estimator `log pi(a_A | s, g) - log pi(a_B | s, g)` and avoids scoring the assistant's full reasoning text.

In [8]:
EXEC_WORKERS  = 16   # concurrent vLLM calls (actor + judge combined)
COORD_WORKERS = 32   # coordinator threads (cheap, mostly blocking on futures)

EXEC_POOL = ThreadPoolExecutor(max_workers=EXEC_WORKERS, thread_name_prefix='vllm')

@dataclass
class PairResult:
    pair_index: int
    preference: str
    gt_sign: int
    guidance: str
    guidance_attempts: int
    guidance_regenerated: bool
    guidance_issues: list
    action_A: str
    action_B: str
    logp_base_A: float
    logp_base_B: float
    logp_ic_A: float
    logp_ic_B: float
    n_tok_A: int
    n_tok_B: int
    delta: float
    pred_sign: int
    match: bool

def estimate_pair(pair: dict, reflector_seed: Optional[int] = None,
                  state_window: Optional[int] = None) -> PairResult:
    h = extract_hindsight_pointwise(pair)

    if USE_JOINT_ACTION_REFLECTOR:
        prompt_text = format_joint_action_prompt(h, state_window=4)
        fut_ref = EXEC_POOL.submit(call_joint_action_reflector, prompt_text)
    else:
        prompt_text, _ = format_pointwise_prompt(h, randomize_order=True, seed=reflector_seed)
        fut_ref = EXEC_POOL.submit(call_pointwise_reflector, prompt_text)

    guidance_info = fut_ref.result()
    guidance = guidance_info['guidance']

    fut_iA = EXEC_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_A'], guidance, state_window)
    fut_iB = EXEC_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_B'], guidance, state_window)
    ic_A = fut_iA.result(); ic_B = fut_iB.result()

    delta = ic_A['mean_logprob'] - ic_B['mean_logprob']
    gt_sign = +1 if pair['preference'] == 'original' else -1
    pred_sign = 1 if delta > 0 else (-1 if delta < 0 else 0)

    return PairResult(
        pair_index=int(pair['pair_index']),
        preference=pair['preference'],
        gt_sign=gt_sign,
        guidance=guidance,
        guidance_attempts=guidance_info['attempts'],
        guidance_regenerated=guidance_info['regenerated'],
        guidance_issues=guidance_info['issues'],
        action_A=h['score_action_A'],
        action_B=h['score_action_B'],
        logp_base_A=float('nan'), logp_base_B=float('nan'),
        logp_ic_A=ic_A['logprob'], logp_ic_B=ic_B['logprob'],
        n_tok_A=ic_A['n_tokens'], n_tok_B=ic_B['n_tokens'],
        delta=delta,
        pred_sign=pred_sign,
        match=(pred_sign == gt_sign),
    )




## Smoke test on one pair

In [9]:
sample = df_eval.iloc[0].to_dict()
smoke_res = estimate_pair(sample, reflector_seed=sample['pair_index'], state_window=SCORING_STATE_WINDOW)
print('pair_index :', smoke_res.pair_index)
print('preference :', smoke_res.preference, ' gt_sign:', smoke_res.gt_sign)
print('delta      :', smoke_res.delta)
print('pred_sign  :', smoke_res.pred_sign, ' match:', smoke_res.match)
print('\n--- guidance ---\n')
print(smoke_res.guidance)

pair_index : 2
preference : original  gt_sign: 1
delta      : -0.00029431381549330227
pred_sign  : -1  match: False

--- guidance ---

Action A tested velocity dependence. Results showed force was consistent across different initial speeds. This confirms velocity independence but leaves the mass and distance scaling unverified at new points.

Action B tests a mass value of 3.0, whereas previous tests used 1, 2, and 4. This probes the mass scaling at an intermediate point. Verifying the trend at a non-power-of-two value reduces the risk of fitting artifacts specific to powers of two.

Action B is superior. The core discovery depends on mass and distance relationships. Action A confirmed a secondary property already suggested by the data. Action B provides critical validation of the mass scaling law by testing an unexplored integer value. With limited rounds, verifying primary scaling parameters at intermediate points yields higher information gain than reconfirming velocity independence

## Run across all non-tie pairs

A coordinator pool dispatches pair-level work; each coordinator is a cheap thread that submits the five vLLM calls into the shared `EXEC_POOL`. With `COORD_WORKERS=32` and `EXEC_WORKERS=16`, up to 16 vLLM requests are in flight at once, globally pipelined across all 48 pairs.

In [10]:
def _run_pair(pair):
    return estimate_pair(pair, reflector_seed=pair['pair_index'], state_window=SCORING_STATE_WINDOW)

print(f"Reflector: {'joint_action' if USE_JOINT_ACTION_REFLECTOR else 'pointwise'}")
print(f"Score target: {'action_block' if SCORE_ACTION_BLOCK_ONLY else 'full_assistant_no_think'}")

results, errors = [], []
pairs_list = [row.to_dict() for _, row in df_eval.iterrows()]

with ThreadPoolExecutor(max_workers=min(COORD_WORKERS, len(pairs_list)), thread_name_prefix='coord') as coord_pool:
    futures = {coord_pool.submit(_run_pair, pair): pair['pair_index'] for pair in pairs_list}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='pairs'):
        pid = futures[fut]
        try:
            results.append(fut.result())
        except Exception as e:
            errors.append((pid, repr(e)))
            print('error on pair', pid, e)

results.sort(key=lambda r: r.pair_index)
res_df = pd.DataFrame([asdict(r) for r in results])
out_path = Path('adv_est_results.jsonl')
res_df.to_json(out_path, orient='records', lines=True)
print(f'saved {len(res_df)} rows to {out_path}; errors={len(errors)}; exec_workers={EXEC_WORKERS} coord_workers={COORD_WORKERS}')




Reflector: joint_action
Score target: action_block


pairs: 100%|████████████████████████████████████| 45/45 [11:03<00:00, 14.75s/it]

saved 45 rows to adv_est_results.jsonl; errors=0; exec_workers=16 coord_workers=32


## Sign-match rate and breakdown

In [11]:
import numpy as np
n = len(res_df)
acc = res_df['match'].mean()
print(f'overall sign-match: {acc:.3f}  (n={n})')
print('\nby ground-truth preference:')
print(res_df.groupby('preference')['match'].agg(['mean', 'count']))
print('\ndelta distribution:')
print(res_df['delta'].describe())
print('\n|delta| on matches vs misses:')
print(res_df.assign(abs_delta=res_df['delta'].abs()).groupby('match')['abs_delta'].describe())

overall sign-match: 0.467  (n=45)

by ground-truth preference:
                    mean  count
preference                     
counterfactual  0.125000     24
original        0.857143     21

delta distribution:
count    45.000000
mean      0.068021
std       0.118168
min      -0.373068
25%       0.018188
50%       0.044508
75%       0.125385
max       0.381498
Name: delta, dtype: float64

|delta| on matches vs misses:
       count      mean       std       min       25%       50%       75%  \
match                                                                      
False   24.0  0.098188  0.097411  0.001888  0.024152  0.054860  0.136495   
True    21.0  0.094847  0.095935  0.000119  0.019973  0.058086  0.141909   

            max  
match            
False  0.381498  
True   0.373068  


In [12]:
# quick bootstrap CI on sign-match
rng = np.random.default_rng(0)
B = 2000
boots = [rng.choice(res_df['match'].values, size=len(res_df), replace=True).mean() for _ in range(B)]
lo, hi = np.quantile(boots, [0.025, 0.975])
print(f'sign-match = {acc:.3f}  95% CI [{lo:.3f}, {hi:.3f}]')

sign-match = 0.467  95% CI [0.333, 0.600]


In [13]:
res_df


,pair_index,preference,gt_sign,guidance,guidance_attempts,guidance_regenerated,guidance_issues,action_A,action_B,logp_base_A,logp_base_B,logp_ic_A,logp_ic_B,n_tok_A,n_tok_B,delta,pred_sign,match
0,2,original,1,Action A tested velocity independence. By vary...,2,True,[],"<run_experiment>\n[\n {""mass1"": 1.0, ""mass2""...","<run_experiment>\n[\n {""mass1"": 1.0, ""mass2""...",NaN,NaN,-11.980905,-18.566142,180,127,0.079630,1,True
1,3,original,1,The experiment varying charge to 1.5 and 2.0 r...,2,True,[],"<run_experiment>\n[\n {""q1"": 1.0, ""m1"": 1e15...","<run_experiment>\n[\n {""q1"": 1.0, ""m1"": 1e15...",NaN,NaN,-2.983275,-19.733620,342,407,0.039763,1,True
2,6,original,1,Action A isolated variables systematically. It...,2,True,"[equation_like, final_law_reference]","<run_experiment>\n[\n {""k"": 1.0, ""A"": 1.0, ""...","<run_experiment>\n[\n {""k"": 1.0, ""A"": 1.0, ""...",NaN,NaN,-5.297240,-9.844004,232,232,0.019598,1,True
3,7,counterfactual,-1,"Action A tests multiple variables—current, mas...",2,True,[],"<run_experiment>\n[\n {""current1"": 1.0, ""cur...","<run_experiment>\n[\n {""current1"": 1.0, ""cur...",NaN,NaN,-12.494393,-21.888589,385,155,0.108764,1,False
4,9,counterfactual,-1,The first action executed a robust scaling swe...,2,True,[],"<run_experiment>\n[\n {""current1"": 1.0, ""cur...","<run_experiment>\n[\n {""current1"": 1.0, ""cur...",NaN,NaN,-13.953336,-20.748800,337,337,0.020165,1,False
5,12,counterfactual,-1,Action A varied `lambda` pairs at fixed time. ...,2,True,[],"<run_experiment>\n[\n {""N0a"": 1000.0, ""N0b"":...","<run_experiment>\n[\n {""N0a"": 1000.0, ""N0b"":...",NaN,NaN,-5.531485,-21.462614,206,203,0.078875,1,False
6,14,original,1,Action A probed mixed charge signs. Results sh...,2,True,[],"<run_experiment>\n[\n {""q1"": 1.0, ""m1"": 1e10...","<run_experiment>\n[\n {""q1"": -2.0, ""m1"": 1e1...",NaN,NaN,-7.582011,-19.690274,352,352,0.034398,1,True
7,15,counterfactual,-1,Action A executed experiments densely near the...,2,True,[],"<run_experiment>\n[\n {""I_0"": 100.0, ""theta""...",<python>\nimport math\n\n# Let me think about ...,NaN,NaN,-27.288030,-570.117180,318,1220,0.381498,1,False
8,16,original,1,Testing extreme omega values of 1e5 and 1e15 e...,2,True,[],"<run_experiment>\n[\n {""omega"": 1e5, ""tempera...","<final_law>\ndef discovered_law(omega, T):\n\t...",NaN,NaN,-3.098606,-9.881283,93,52,0.156706,1,True
9,20,counterfactual,-1,Action A varied currents from 1.0 to 3.0 at a ...,2,True,[],"<run_experiment>\n[\n {""current1"": 1.0, ""cur...","<run_experiment>\n[\n {""current1"": 2.0, ""cur...",NaN,NaN,-6.626576,-9.298942,332,206,0.025181,1,False


In [14]:
# Posthoc filtering by action type and base margin
REQUIRE_SAME_ACTION_TYPE = True
BASE_MARGIN_QUANTILES = [0.5, 0.6, 0.7, 0.8]
SAVE_AUGMENTED_RESULTS = False


def action_type(action_text: str) -> str:
    m = re.search(r'<(run_experiment|python|final_law)>', action_text or '')
    return m.group(1) if m else 'other'


def bootstrap_ci(matches, B: int = 2000, seed: int = 0):
    matches = np.asarray(matches, dtype=float)
    if len(matches) == 0:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    boots = [rng.choice(matches, size=len(matches), replace=True).mean() for _ in range(B)]
    return np.quantile(boots, [0.025, 0.975])


def summarize_subset(label: str, df_subset: pd.DataFrame, total_n: int):
    n = len(df_subset)
    if n == 0:
        print(f'{label:28s} n=0')
        return
    acc = df_subset['match'].mean()
    lo, hi = bootstrap_ci(df_subset['match'].values)
    print(f'{label:28s} n={n:3d} retained={n / total_n:6.1%} sign-match={acc:.3f} 95% CI [{lo:.3f}, {hi:.3f}]')


def compute_base_scores(pair: dict) -> dict:
    h = extract_hindsight_pointwise(pair)
    fut_A = EXEC_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_A'], None, SCORING_STATE_WINDOW)
    fut_B = EXEC_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_B'], None, SCORING_STATE_WINDOW)
    base_A = fut_A.result()
    base_B = fut_B.result()
    return {
        'pair_index': int(pair['pair_index']),
        'base_mean_logp_A': base_A['mean_logprob'],
        'base_mean_logp_B': base_B['mean_logprob'],
        'base_n_tok_A': base_A['n_tokens'],
        'base_n_tok_B': base_B['n_tokens'],
        'base_margin': base_A['mean_logprob'] - base_B['mean_logprob'],
    }


need_base_recompute = (
    'base_margin' not in res_df.columns
    or 'base_mean_logp_A' not in res_df.columns
    or 'base_mean_logp_B' not in res_df.columns
    or res_df['base_margin'].isna().all()
)

if need_base_recompute:
    print('Recomputing unguided base logprobs for base-margin filtering...')
    pairs_list = [row.to_dict() for _, row in df_eval.iterrows()]
    base_rows = []
    with ThreadPoolExecutor(max_workers=min(COORD_WORKERS, len(pairs_list)), thread_name_prefix='base-coord') as coord_pool:
        futures = {coord_pool.submit(compute_base_scores, pair): pair['pair_index'] for pair in pairs_list}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='base logprobs'):
            base_rows.append(fut.result())
    base_df = pd.DataFrame(base_rows)
    res_df = res_df.drop(columns=['base_mean_logp_A', 'base_mean_logp_B', 'base_n_tok_A', 'base_n_tok_B', 'base_margin'], errors='ignore')
    res_df = res_df.merge(base_df, on='pair_index', how='left')
else:
    print('Using existing base logprobs already present in res_df.')

res_df['action_type_A'] = res_df['action_A'].map(action_type)
res_df['action_type_B'] = res_df['action_B'].map(action_type)
res_df['same_action_type'] = res_df['action_type_A'] == res_df['action_type_B']
res_df['abs_base_margin'] = res_df['base_margin'].abs()

if SAVE_AUGMENTED_RESULTS:
    augmented_out = Path('adv_est_results_with_base.jsonl')
    res_df.to_json(augmented_out, orient='records', lines=True)
    print(f'Saved augmented results to {augmented_out}')

print(f'SCORE_ACTION_BLOCK_ONLY = {SCORE_ACTION_BLOCK_ONLY}')
print(f'REQUIRE_SAME_ACTION_TYPE = {REQUIRE_SAME_ACTION_TYPE}')
print()

all_df = res_df.copy()
summarize_subset('all pairs', all_df, total_n=len(res_df))

same_type_df = all_df[all_df['same_action_type']].copy() if REQUIRE_SAME_ACTION_TYPE else all_df.copy()
label = 'same action type' if REQUIRE_SAME_ACTION_TYPE else 'no type filter'
summarize_subset(label, same_type_df, total_n=len(res_df))

if len(same_type_df) == 0:
    print('No retained pairs after action-type filtering.')
else:
    valid_margin_df = same_type_df.dropna(subset=['abs_base_margin']).copy()
    if len(valid_margin_df) == 0:
        print('No valid base margins available.')
    else:
        print('base-margin sweep within retained pairs:')
        for q in BASE_MARGIN_QUANTILES:
            tau = valid_margin_df['abs_base_margin'].quantile(q)
            filtered_df = valid_margin_df[valid_margin_df['abs_base_margin'] <= tau]
            summarize_subset(f'abs(base_margin) <= q{q:.1f}', filtered_df, total_n=len(res_df))
            print(f'  tau = {tau:.4f}')



Recomputing unguided base logprobs for base-margin filtering...


base logprobs: 100%|████████████████████████████| 45/45 [04:32<00:00,  6.05s/it]

SCORE_ACTION_BLOCK_ONLY = True
REQUIRE_SAME_ACTION_TYPE = True

all pairs                    n= 45 retained=100.0% sign-match=0.467 95% CI [0.333, 0.600]
same action type             n= 27 retained= 60.0% sign-match=0.444 95% CI [0.259, 0.630]
base-margin sweep within retained pairs:
abs(base_margin) <= q0.5     n= 14 retained= 31.1% sign-match=0.500 95% CI [0.214, 0.786]
  tau = 0.0147
abs(base_margin) <= q0.6     n= 16 retained= 35.6% sign-match=0.500 95% CI [0.250, 0.750]
  tau = 0.0263
abs(base_margin) <= q0.7     n= 19 retained= 42.2% sign-match=0.526 95% CI [0.316, 0.737]
  tau = 0.0410
abs(base_margin) <= q0.8     n= 21 retained= 46.7% sign-match=0.524 95% CI [0.333, 0.762]
  tau = 0.0492


In [15]:
# Compare unguided actor preference vs reflected preference
DEFAULT_BASE_MARGIN_QUANTILES = [0.5, 0.6, 0.7, 0.8]


def _sign(x: float) -> int:
    if pd.isna(x):
        return 0
    return 1 if x > 0 else (-1 if x < 0 else 0)


def _bootstrap_ci(matches, B: int = 2000, seed: int = 0):
    matches = np.asarray(matches, dtype=float)
    if len(matches) == 0:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    boots = [rng.choice(matches, size=len(matches), replace=True).mean() for _ in range(B)]
    return np.quantile(boots, [0.025, 0.975])


def _ensure_action_type_cols(df: pd.DataFrame) -> pd.DataFrame:
    if 'same_action_type' in df.columns:
        return df

    def action_type(action_text: str) -> str:
        m = re.search(r'<(run_experiment|python|final_law)>', action_text or '')
        return m.group(1) if m else 'other'

    df = df.copy()
    df['action_type_A'] = df['action_A'].map(action_type)
    df['action_type_B'] = df['action_B'].map(action_type)
    df['same_action_type'] = df['action_type_A'] == df['action_type_B']
    return df


def summarize_base_vs_reflected(label: str, df_subset: pd.DataFrame, total_n: int):
    n = len(df_subset)
    if n == 0:
        print(f'{label:28s} n=0')
        return

    base_match = (df_subset['base_pred_sign'] == df_subset['gt_sign'])
    refl_match = (df_subset['pred_sign'] == df_subset['gt_sign'])
    base_acc = base_match.mean()
    refl_acc = refl_match.mean()
    base_lo, base_hi = _bootstrap_ci(base_match.values)
    refl_lo, refl_hi = _bootstrap_ci(refl_match.values)

    helped = ((~base_match) & refl_match).sum()
    hurt = (base_match & (~refl_match)).sum()
    no_flip = (df_subset['base_pred_sign'] == df_subset['pred_sign']).sum()
    flipped_no_gain = ((df_subset['base_pred_sign'] != df_subset['pred_sign']) & (~base_match) & (~refl_match)).sum()

    print(f'{label:28s} n={n:3d} retained={n / total_n:6.1%}')
    print(f'  base sign-match      = {base_acc:.3f} 95% CI [{base_lo:.3f}, {base_hi:.3f}]')
    print(f'  reflected sign-match = {refl_acc:.3f} 95% CI [{refl_lo:.3f}, {refl_hi:.3f}]')
    print(f'  reflection helped    = {helped:3d}')
    print(f'  reflection hurt      = {hurt:3d}')
    print(f'  no sign flip         = {no_flip:3d}')
    print(f'  flip, still wrong    = {flipped_no_gain:3d}')


if 'base_margin' not in res_df.columns:
    raise ValueError('base_margin not found in res_df. Run the previous filtering cell first.')

cmp_df = _ensure_action_type_cols(res_df).copy()
cmp_df['base_pred_sign'] = cmp_df['base_margin'].map(_sign)
cmp_df['abs_base_margin'] = cmp_df['base_margin'].abs()
quantiles = globals().get('BASE_MARGIN_QUANTILES', DEFAULT_BASE_MARGIN_QUANTILES)

print(f'SCORE_ACTION_BLOCK_ONLY = {globals().get("SCORE_ACTION_BLOCK_ONLY", "<unknown>")}')
print(f'REQUIRE_SAME_ACTION_TYPE = {globals().get("REQUIRE_SAME_ACTION_TYPE", "<unknown>")}')
print()

summarize_base_vs_reflected('all pairs', cmp_df, total_n=len(cmp_df))
print()

same_type_df = cmp_df[cmp_df['same_action_type']].copy()
summarize_base_vs_reflected('same action type', same_type_df, total_n=len(cmp_df))

if len(same_type_df) > 0:
    valid_margin_df = same_type_df.dropna(subset=['abs_base_margin']).copy()
    if len(valid_margin_df) > 0:
        print('Base-margin sweep within same-type pairs:')
        for q in quantiles:
            tau = valid_margin_df['abs_base_margin'].quantile(q)
            filtered_df = valid_margin_df[valid_margin_df['abs_base_margin'] <= tau]
            summarize_base_vs_reflected(f'abs(base_margin) <= q{q:.1f}', filtered_df, total_n=len(cmp_df))
            print(f'  tau = {tau:.4f}')
            print()



SCORE_ACTION_BLOCK_ONLY = True
REQUIRE_SAME_ACTION_TYPE = True

all pairs                    n= 45 retained=100.0%
  base sign-match      = 0.467 95% CI [0.333, 0.622]
  reflected sign-match = 0.467 95% CI [0.333, 0.600]
  reflection helped    =   1
  reflection hurt      =   1
  no sign flip         =  43
  flip, still wrong    =   0

same action type             n= 27 retained= 60.0%
  base sign-match      = 0.481 95% CI [0.296, 0.667]
  reflected sign-match = 0.444 95% CI [0.259, 0.630]
  reflection helped    =   0
  reflection hurt      =   1
  no sign flip         =  26
  flip, still wrong    =   0
Base-margin sweep within same-type pairs:
abs(base_margin) <= q0.5     n= 14 retained= 31.1%
  base sign-match      = 0.571 95% CI [0.286, 0.857]
  reflected sign-match = 0.500 95% CI [0.214, 0.786]
  reflection helped    =   0
  reflection hurt      =   1
  no sign flip         =  13
  flip, still wrong    =   0
  tau = 0.0147

abs(base_margin) <= q0.6     n= 16 retained= 35.6%
  base 

In [16]:
cmp_df['gt_prefers_original'] = (cmp_df['preference'] == 'original')
summarize_base_vs_reflected('GT prefers original (= A)',
    cmp_df[cmp_df['gt_prefers_original']], total_n=len(cmp_df))
summarize_base_vs_reflected('GT prefers counterfactual (= B)',
    cmp_df[~cmp_df['gt_prefers_original']], total_n=len(cmp_df))

GT prefers original (= A)    n= 21 retained= 46.7%
  base sign-match      = 0.857 95% CI [0.714, 1.000]
  reflected sign-match = 0.857 95% CI [0.714, 1.000]
  reflection helped    =   0
  reflection hurt      =   0
  no sign flip         =  21
  flip, still wrong    =   0
GT prefers counterfactual (= B) n= 24 retained= 53.3%
  base sign-match      = 0.125 95% CI [0.000, 0.250]
  reflected sign-match = 0.125 95% CI [0.000, 0.250]
  reflection helped    =   1
  reflection hurt      =   1
  no sign flip         =  22
  flip, still wrong    =   0
